# NB24 · Context Managers

`advanced/` · 20 questions · the last one is a story project

The `with` statement is how Python promises "this gets cleaned up, no
matter what". Here you look under the hood: write your own managers with
`__enter__` / `__exit__` and with `@contextmanager`, decide what happens
to exceptions, and use `contextlib` helpers. Files, timers, settings and
a tiny database transaction are the examples.

## You'll need

- NB23: decorators
- NB12: exceptions

## You'll practise

- The with protocol: __enter__ and __exit__
- Exception info and suppression (returning True)
- contextlib.contextmanager, suppress, redirect_stdout, closing, nullcontext
- ExitStack for a varying number of resources
- Patterns: temporary changes, timers, transactions
- Parenthesised multi-item with (3.10+)

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Real-life

`path` is a notes file. Read it with a `with` block and print its first
line, then print whether the file object is closed once the block has ended.

**Expected output:**

```text
buy milk
True
```

<details><summary><b>Solution</b></summary>

```python
with open(path) as f:
    first = f.readline().strip()
print(first)
print(f.closed)
```

</details>

In [ ]:
import os, tempfile
folder = tempfile.TemporaryDirectory()
path = os.path.join(folder.name, "notes.txt")
with open(path, "w") as f:
    f.write("buy milk\ncall Sam\n")

# your code here


### Q02 · Real-life

Remove `"coupon"` from the `cart` dict, but the key may not be there.
Use `contextlib.suppress` so a missing key is ignored, then print the cart.

**Expected output:**

```text
{'apples': 3, 'bread': 1}
```

<details><summary><b>Solution</b></summary>

```python
from contextlib import suppress
with suppress(KeyError):
    del cart["coupon"]
print(cart)
```

</details>

In [ ]:
cart = {"apples": 3, "bread": 1}

# your code here


### Q03 · Output-match

The function `noisy()` prints a line. Capture what it prints into a string
with `redirect_stdout`, then print the captured text in upper case and its
length (the length includes the newline).

**Expected output:**

```text
LOW DISK SPACE
15
```

<details><summary><b>Solution</b></summary>

```python
import io
from contextlib import redirect_stdout
buffer = io.StringIO()
with redirect_stdout(buffer):
    noisy()
text = buffer.getvalue()
print(text.upper().strip())
print(len(text))
```

</details>

In [ ]:
def noisy():
    print("low disk space")

# your code here


### Q04 · Build

Write a class `Door` with `__enter__` and `__exit__` so that
`with Door(): print("walking through")` prints `door opens`, then the
line from inside the block, then `door closes`.

**Expected output:**

```text
door opens
walking through
door closes
```

<details><summary><b>Solution</b></summary>

```python
class Door:
    def __enter__(self):
        print("door opens")
        return self
    def __exit__(self, exc_type, exc, tb):
        print("door closes")
        return False

with Door():
    print("walking through")
```

</details>

In [ ]:
# your code here


### Q05 · Real-life

`get_title` should work with or without a lock. `lock` is either a real
context manager or `None`. Use `contextlib.nullcontext` so one `with`
line handles both cases, and print the title.

**Expected output:**

```text
Dune
```

<details><summary><b>Solution</b></summary>

```python
manager = lock if lock is not None else nullcontext()
with manager:
    print(get_title())
```

</details>

In [ ]:
from contextlib import nullcontext
class Lock:
    def __enter__(self):
        print("locked")
    def __exit__(self, *args):
        print("unlocked")

lock = None
def get_title():
    return "Dune"

# your code here


---

## Combine · 2 to 4 lines

### Q06 · Build

With `@contextmanager`, write `banner(title)`: it prints `== title ==`
on entering and `== end ==` on leaving. Use it around two prints of your
choice.

**Expected output:**

```text
== Report ==
sales: 120
returns: 4
== end ==
```

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def banner(title):
    print("==", title, "==")
    yield
    print("== end ==")

with banner("Report"):
    print("sales: 120")
    print("returns: 4")
```

</details>

In [ ]:
# your code here


### Q07 · Real-life

Write a `Timer` class (context manager) that stores `elapsed` on itself:
the clock reading on exit minus the reading on entry. `fake_clock()` plays
the part of a real clock so the numbers are stable. Time a block and print
the elapsed seconds.

**Expected output:**

```text
Elapsed: 2.5
```

<details><summary><b>Solution</b></summary>

```python
class Timer:
    def __enter__(self):
        self.start = fake_clock()
        return self
    def __exit__(self, exc_type, exc, tb):
        self.elapsed = fake_clock() - self.start
        return False

with Timer() as t:
    total = sum(range(1000))
print("Elapsed:", t.elapsed)
```

</details>

In [ ]:
ticks = iter([10.0, 12.5])
def fake_clock():
    return next(ticks)

# your code here


### Q08 · Puzzle

Write a class `Quiet` whose `__exit__` swallows any exception and
remembers its type name in `self.seen`. Inside `with Quiet() as q:` raise
`ValueError("boom")`, then print `survived` and the remembered name. Does
the line after the raise still run inside the block?

**Expected output:**

```text
survived
ValueError
```

<details><summary><b>Solution</b></summary>

```python
class Quiet:
    def __enter__(self):
        self.seen = None
        return self
    def __exit__(self, exc_type, exc, tb):
        self.seen = exc_type.__name__
        return True

with Quiet() as q:
    raise ValueError("boom")
    print("never printed")
print("survived")
print(q.seen)
```

</details>

In [ ]:
# your code here


### Q09 · Real-life

Write `@contextmanager` function `temporarily(settings, key, value)`: it
sets `settings[key]` to `value` for the duration of the block and puts
the old value back afterwards. Print the settings inside and after.

**Expected output:**

```text
{'theme': 'dark', 'font': 12}
{'theme': 'light', 'font': 12}
```

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def temporarily(settings, key, value):
    old = settings[key]
    settings[key] = value
    try:
        yield settings
    finally:
        settings[key] = old

with temporarily(settings, "theme", "dark"):
    print(settings)
print(settings)
```

</details>

In [ ]:
settings = {"theme": "light", "font": 12}

# your code here


### Q10 · Real-life

`Connection` has a `close()` method but is not a context manager. Use
`contextlib.closing` to wrap `Connection("db1")`, print its `name` inside
the block, and let the block end.

**Expected output:**

```text
using db1
closing db1
```

<details><summary><b>Solution</b></summary>

```python
from contextlib import closing
with closing(Connection("db1")) as conn:
    print("using", conn.name)
```

</details>

In [ ]:
class Connection:
    def __init__(self, name):
        self.name = name
    def close(self):
        print("closing", self.name)

# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Real-life

Write every name in `names` to its own file inside `folder`
(`<name>.txt`, containing the name). Open all the files at once with
`ExitStack`, write through them, and after the `with` block print the
list of `closed` flags, one per file.

**Expected output:**

```text
[True, True, True]
```

<details><summary><b>Solution</b></summary>

```python
from contextlib import ExitStack
with ExitStack() as stack:
    files = [stack.enter_context(open(os.path.join(folder.name, name + ".txt"), "w"))
             for name in names]
    for name, f in zip(names, files):
        f.write(name)
print([f.closed for f in files])
```

</details>

In [ ]:
import os, tempfile
folder = tempfile.TemporaryDirectory()
names = ["ada", "grace", "linus"]

# your code here


### Q12 · Real-life

`read_price(prices, item)` should treat a **missing item** as "no price"
but must **not** hide real bugs. Use `suppress(KeyError)` inside it, then
call it once with a known item and once with an unknown item. Finally call
`read_price(prices, 7)` too: print the name of the error that escapes,
caught with a normal `try`.

**Expected output:**

```text
2.5
None
AttributeError
```

<details><summary><b>Solution</b></summary>

```python
from contextlib import suppress

def read_price(prices, item):
    price = None
    with suppress(KeyError):
        price = lookup(prices, item)
    return price

print(read_price(prices, "Tea"))
print(read_price(prices, "Pizza"))
try:
    read_price(prices, 7)
except AttributeError as e:
    print(type(e).__name__)
```

</details>

In [ ]:
prices = {"tea": 2.5, "cake": 4.0}
def lookup(prices, item):
    return prices[item.lower()]

# your code here


### Q13 · Real-life

A `Transaction` class: on entering, it prints `begin`; on leaving it
prints `commit` if all went well or `rollback: <error message>` if an
exception happened, and in the second case it must **not** swallow the
exception. Run a failing block (raise `RuntimeError("disk full")`) inside
a normal `try`, and print `caught` afterwards. Also run a clean block.

**Expected output:**

```text
begin
saving
commit
begin
rollback: disk full
caught
```

<details><summary><b>Solution</b></summary>

```python
class Transaction:
    def __enter__(self):
        print("begin")
        return self
    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            print("commit")
        else:
            print("rollback:", exc)
        return False

with Transaction():
    print("saving")
try:
    with Transaction():
        raise RuntimeError("disk full")
except RuntimeError:
    print("caught")
```

</details>

In [ ]:
# your code here


### Q14 · Real-life

`report()` prints a few log lines to the screen. Capture its output with
`redirect_stdout`, then print how many lines it produced and only the
lines containing `ERROR`, each prefixed with `> `.

**Expected output:**

```text
4 lines
> ERROR disk
> ERROR net
```

<details><summary><b>Solution</b></summary>

```python
import io
from contextlib import redirect_stdout
buffer = io.StringIO()
with redirect_stdout(buffer):
    report()
lines = buffer.getvalue().splitlines()
print(len(lines), "lines")
for line in lines:
    if "ERROR" in line:
        print("> " + line)
```

</details>

In [ ]:
def report():
    print("INFO start")
    print("ERROR disk")
    print("INFO retry")
    print("ERROR net")

# your code here


### Q15 · Real-life

Copy `source.txt` to `copy.txt` inside `folder`, changing every line to
upper case. Open both files in **one** `with` statement using the
parenthesised multi-item form. Then print the contents of the copy.

**Expected output:**

```text
ALPHA
BETA
GAMMA
```

<details><summary><b>Solution</b></summary>

```python
with (
    open(src) as reader,
    open(dst, "w") as writer,
):
    for line in reader:
        writer.write(line.upper())
with open(dst) as f:
    print(f.read())
```

</details>

In [ ]:
import os, tempfile
folder = tempfile.TemporaryDirectory()
src = os.path.join(folder.name, "source.txt")
dst = os.path.join(folder.name, "copy.txt")
with open(src, "w") as f:
    f.write("alpha\nbeta\ngamma\n")

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Puzzle

Write class `Tracer(name, log, swallow=False)`: `__enter__` appends
`"enter <name>"` to `log`; `__exit__` appends `"exit <name>"` and returns
`swallow`. Run
`with Tracer("a", log, swallow=True), Tracer("b", log), Tracer("c", log):`
with a body that appends `"body"` and then raises `ValueError`. Print the
log, one entry per line, then print `done` (the error must not escape).

**Expected output:**

```text
enter a
enter b
enter c
body
exit c
exit b
exit a
done
```

<details><summary><b>Solution</b></summary>

```python
class Tracer:
    def __init__(self, name, log, swallow=False):
        self.name = name
        self.log = log
        self.swallow = swallow
    def __enter__(self):
        self.log.append("enter " + self.name)
        return self
    def __exit__(self, exc_type, exc, tb):
        self.log.append("exit " + self.name)
        return self.swallow

with Tracer("a", log, swallow=True), Tracer("b", log), Tracer("c", log):
    log.append("body")
    raise ValueError
for entry in log:
    print(entry)
print("done")
```

</details>

In [ ]:
log = []

# your code here


### Q17 · Real-life

Write `@contextmanager` `recovering(log)`: if the block raises
`ValueError`, append `"recovered: <message>"` to `log` and let the program
carry on; any other error must still escape. Run a block that raises
`ValueError("bad row")`, a clean block, and a block that raises
`KeyError("x")` (catch that one with a normal `try`). Print the log and
which blocks reached the line after them.

**Expected output:**

```text
after block 1
after block 2
block 3 escaped
['recovered: bad row', 'fine']
```

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def recovering(log):
    try:
        yield
    except ValueError as e:
        log.append("recovered: " + str(e))

with recovering(log):
    raise ValueError("bad row")
print("after block 1")
with recovering(log):
    log.append("fine")
print("after block 2")
try:
    with recovering(log):
        raise KeyError("x")
except KeyError:
    print("block 3 escaped")
print(log)
```

</details>

In [ ]:
log = []

# your code here


### Q18 · Puzzle

`acquire(name)` returns a resource and prints `got <name>`, except for
`"printer"` which raises `RuntimeError`. Each resource has `release()`
that prints `released <name>`. Using one `ExitStack` and
`stack.callback(...)`, acquire `["disk", "network", "printer", "camera"]`
in order inside a `try`, registering a release right after each success.
When the error happens, print `failed: <message>` after the cleanups run.
In which order do the releases happen?

**Expected output:**

```text
got disk
got network
released network
released disk
failed: printer is offline
```

<details><summary><b>Solution</b></summary>

```python
from contextlib import ExitStack
try:
    with ExitStack() as stack:
        for name in ["disk", "network", "printer", "camera"]:
            resource = acquire(name)
            stack.callback(resource.release)
except RuntimeError as e:
    print("failed:", e)
```

</details>

In [ ]:
class Resource:
    def __init__(self, name):
        self.name = name
    def release(self):
        print("released", self.name)

def acquire(name):
    if name == "printer":
        raise RuntimeError("printer is offline")
    print("got", name)
    return Resource(name)

# your code here


### Q19 · Build

Build a tiny pretty-printer for nested outlines. `indent()` is a
`@contextmanager` that raises the current depth by one while its block
runs (keep the depth in a dict or list so the function `say(text)` can
see it). `say(text)` prints the text with two spaces per level of depth.
Print an outline: `project`, then (indented) `src`, then (deeper) `main.py`
and `utils.py`, then (back at depth 1) `README`.

**Expected output:**

```text
project
  src
    main.py
    utils.py
  README
(end)
```

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

state = {"depth": 0}

@contextmanager
def indent():
    state["depth"] += 1
    try:
        yield
    finally:
        state["depth"] -= 1

def say(text):
    print("  " * state["depth"] + text)

say("project")
with indent():
    say("src")
    with indent():
        say("main.py")
        say("utils.py")
    say("README")
say("(end)")
```

</details>

In [ ]:
# your code here


---

## Q20 · Story project: Bank Vault Transactions

*Pixel Credit Union* keeps its accounts in a plain dict, and a crash halfway
through a transfer once left money in neither account. You'll build a small
transaction manager so changes to the "database" **roll back** whenever
something goes wrong. Each step builds on the one before.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
import copy
from contextlib import ExitStack, contextmanager

class Database:
    def __init__(self, data):
        self.data = data

bank = Database({"ada": 100, "grace": 50})


### Step 1

Write a `Transaction` class (context manager) that takes a `Database`. On entering, it saves a deep copy of `db.data`. On a clean exit it prints `commit`. If the block raised, it restores the saved data **in place** (clear the dict and update it, so other references stay valid), prints `rollback`, and lets the exception continue. Test it: one block that sets `ada` to 70 cleanly, and one that sets `ada` to 0 and then raises `RuntimeError` (catch it with a normal `try`). Print `bank.data` after each.

**Expected output:**

```text
commit
{'ada': 70, 'grace': 50}
rollback
{'ada': 70, 'grace': 50}
```

<details><summary><b>Solution</b></summary>

```python
class Transaction:
    def __init__(self, db):
        self.db = db
    def __enter__(self):
        self.saved = copy.deepcopy(self.db.data)
        return self.db
    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            print("commit")
        else:
            self.db.data.clear()
            self.db.data.update(self.saved)
            print("rollback")
        return False

with Transaction(bank) as db:
    db.data["ada"] = 70
print(bank.data)
try:
    with Transaction(bank) as db:
        db.data["ada"] = 0
        raise RuntimeError("crash")
except RuntimeError:
    pass
print(bank.data)
```

</details>

In [ ]:
# your code here


### Step 2

Add `class InsufficientFunds(Exception): pass` and a function `transfer(db, source, target, amount)`. Inside a `Transaction`, it first **subtracts** the amount from the source, then adds it to the target; if the source balance went negative it raises `InsufficientFunds` **after** the subtraction. Run a transfer of 20 from `ada` to `grace`, then a transfer of 500 from `grace` to `ada` (catch the error and print `refused`). Print the balances after each.

**Expected output:**

```text
commit
{'ada': 50, 'grace': 70}
rollback
refused
{'ada': 50, 'grace': 70}
```

<details><summary><b>Solution</b></summary>

```python
class InsufficientFunds(Exception):
    pass

def transfer(db, source, target, amount):
    with Transaction(db):
        db.data[source] -= amount
        if db.data[source] < 0:
            raise InsufficientFunds(source)
        db.data[target] += amount

transfer(bank, "ada", "grace", 20)
print(bank.data)
try:
    transfer(bank, "grace", "ada", 500)
except InsufficientFunds:
    print("refused")
print(bank.data)
```

</details>

In [ ]:
# your code here


### Step 3

Transactions can nest. Show it: in an outer `Transaction`, deposit 10 into `ada`; then start an inner `Transaction` that deposits 1000 into `grace` and raises `RuntimeError`, catching that error **inside** the outer block so the outer one still commits. Print the balances at the end: the outer deposit stays, the inner one is gone.

**Expected output:**

```text
rollback
inner caught
commit
{'ada': 60, 'grace': 70}
```

<details><summary><b>Solution</b></summary>

```python
with Transaction(bank) as outer:
    outer.data["ada"] += 10
    try:
        with Transaction(bank) as inner:
            inner.data["grace"] += 1000
            raise RuntimeError("inner failed")
    except RuntimeError:
        print("inner caught")
print(bank.data)
```

</details>

In [ ]:
# your code here


### Step 4

Add an audit trail. Write a `@contextmanager` function `audited(log, label)` that appends `label + ": start"` to `log` before the block and `label + ": ok"` after it. If the block raised, it should append `label + ": failed"` and re-raise (use `try`/`except Exception`/`raise`). Run two transfers (a good 10 from `ada` to `grace`, and a bad 900 from `ada` to `grace`) each wrapped in `audited` using the parenthesised form together with a `Transaction`. Print the log.

**Expected output:**

```text
commit
rollback
t1: start
t1: ok
t2: start
t2: failed
{'ada': 50, 'grace': 80}
```

<details><summary><b>Solution</b></summary>

```python
log = []

@contextmanager
def audited(log, label):
    log.append(label + ": start")
    try:
        yield
    except Exception:
        log.append(label + ": failed")
        raise
    else:
        log.append(label + ": ok")

for label, amount in [("t1", 10), ("t2", 900)]:
    try:
        with (
            audited(log, label),
            Transaction(bank) as db,
        ):
            db.data["ada"] -= amount
            if db.data["ada"] < 0:
                raise InsufficientFunds("ada")
            db.data["grace"] += amount
    except InsufficientFunds:
        pass
for entry in log:
    print(entry)
print(bank.data)
```

</details>

In [ ]:
# your code here


### Step 5

Finally, an all-or-nothing change across **two** databases. Write `update_all(databases, changes)` where `changes` is a list of `(account, delta)` pairs, one per database. Use an `ExitStack` and `enter_context(Transaction(db))` for each database, then apply each change; a balance that would go below zero raises `InsufficientFunds`. Make a second `Database({"ada": 5})` called `savings`. Run one update that works (`+10` on each) and one where the second database fails (`-5` on `bank`'s ada, `-50` on `savings`' ada), and print both databases after each run to prove that the failed run changed neither.

**Expected output:**

```text
commit
commit
{'ada': 60, 'grace': 80} {'ada': 15}
rollback
rollback
batch refused
{'ada': 60, 'grace': 80} {'ada': 15}
```

<details><summary><b>Solution</b></summary>

```python
def update_all(databases, changes):
    with ExitStack() as stack:
        for db in databases:
            stack.enter_context(Transaction(db))
        for db, (account, delta) in zip(databases, changes):
            db.data[account] += delta
            if db.data[account] < 0:
                raise InsufficientFunds(account)

savings = Database({"ada": 5})
update_all([bank, savings], [("ada", 10), ("ada", 10)])
print(bank.data, savings.data)
try:
    update_all([bank, savings], [("ada", -5), ("ada", -50)])
except InsufficientFunds:
    print("batch refused")
print(bank.data, savings.data)
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB25 · Descriptors Metaprogramming** in `advanced/`.